### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/13_CIFAR10_Joint_HPO.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Fnotebooks%2Fthesis%2F13_CIFAR10_Joint_HPO.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Fnotebooks%2Fthesis%2F13_CIFAR10_Joint_HPO.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/13_CIFAR10_Joint_HPO.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.


See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.

Kaggle import fallback: open Kaggle Code → Import Notebook → Local file, upload this notebook, then choose Quick Save.


In [1]:
# Shared setup: use the local initializer when available, otherwise download it.
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "hosted"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = True  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME,
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "notebooks" / "init.py" for path in _locations
                     if (path / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime=RUNTIME, 
    cuda=CUDA
)

Repository ready: /workspace/Continual-Learning-with-Diffusion-Vision-Transformers (runtime: hosted)
Runtime packages ready: Python 3.12.3, TensorFlow 2.20.0, Keras 3.11.2


# CIFAR10: joint diffusion/classifier HPO

All classes train together using **V1 only**, a `new_weight` class token, **raw weights (EMA off)**,
and no distillation. Classifier training searches batch allocation, clean/noisy images and
conditioning labels, using shared internal features and a fixed classifier loss coefficient of 1.0.
The shared API maximizes **ordinary clean-image accuracy** and minimizes **noise loss**
as separate Optuna objectives.

The version-13 notebook retains the version-12 shared profile and the current TMCL route's supported architecture/runtime settings:
`float32`, `fixed-standardize` preprocessing, and a positive classifier MLP ratio.
CNN patchification, cosine LR, V1, batch 128, and native `modify_first_t=False` are fixed;
dropout and classifier stochastic depth each search `[0.0, 0.15, 0.25]`.
This is offline joint HPO; transferring a selected architecture to TMCL also requires its continual protocol.
The revised recipe addresses observed training limitations; it does not guarantee an accuracy target.

**Selection data:** fit on a stratified **80% of the official training set** and use
the remaining **20% for final HPO evaluation**. Official test rows are excluded from
HPO selection. Validation is disabled during fitting; each completed trial is scored
after its final epoch. The explicit `"test"` option remains exploratory only and requires
a separate study directory. Settings selected using official test results cannot support
independent confirmation on those same rows, even with fresh seeds. Preserve HPO
selection metadata when transferring settings. This search is separate from the frozen campaign.

See [the HPO guide](JOINT_CLASSIFIER_HPO.md) for the complete recipe, report findings, and limitations.

In [ ]:
from common.hpo import run_hpo, summarize_hpo
from common.hpo_profiles import JOINT_CLASSIFIER_SEARCH_SPACE
from notebooks.thesis.workflow import check_runtime


print(check_runtime())
JOINT_CLASSIFIER_SEARCH_SPACE

## 1. Budget and concurrency

Batch 128 is fixed in the shared profile and is a starting point for A100/H100, not a measured memory guarantee.
Both notebooks allocate **30 trials with 5 startup trials**, followed by adaptive TPE suggestions.
The earlier wall-clock timeout is intentionally removed (`TIMEOUT_HOURS=None`) so the
study is controlled by its trial allocation and can reach adaptive suggestions.
Failed/pruned attempts still count against the total; external runtime limits can interrupt a study.

Finite V1 fits run all **50 joint epochs**. Early stopping is disabled (`patience=0`),
so a stalled accuracy metric cannot stop training or restore epoch-one weights.
Per-epoch validation is disabled (`validation_freq=[]`) to avoid repeated evaluation during fitting.
Final HPO evaluation remains enabled. Plateau LR callbacks and gradient clipping remain disabled;
the cosine LR schedule is fixed.

Run All continues the remaining **total allocated trial budget**.
Preserve the results directory between hosted sessions. Change `STUDY_ROOT` if scientific settings change.
Version 13 uses a fresh directory because selection data changed. Existing v6-v12 artifacts are preserved.
Set `CONCURRENT_TRIALS` below: **1** trains serially in this kernel; **2** runs up to two
isolated Python training workers on the same GPU. This notebook coordinates Optuna;
no additional notebook kernels are needed. Only one coordinator may open a study.
`WORKER_GPU_MEMORY_LIMIT_MB=None` enables memory growth in each worker; a positive
value caps each worker before GPU initialization. Two workers are a starting point,
not a measured memory or speed guarantee. If this kernel previously trained on the
GPU, restart it before running concurrent trials to release its GPU reservations.
Interrupting the run stops its active workers. Trial logs and artifacts remain available.

In [ ]:
DATASET = "cifar10"
VALIDATION_SOURCE = "split"  # HPO selection uses training-only validation.
VALIDATION_RATIO = 0.2  # Stratified held-out fraction of official training rows.
SEED = 17
TOTAL_TRIALS = 30
CONCURRENT_TRIALS = 2  # 1: serial; 2+: separate training workers on this GPU.
WORKER_GPU_MEMORY_LIMIT_MB = None  # Optional per-worker GPU cap in MB; None uses memory growth.
STARTUP_TRIALS = 5
EPOCHS = 50
DTYPE_POLICY = "float32"
TIMEOUT_HOURS = None  # Trial-count controlled; allow adaptive TPE trials.
STUDY_ROOT = ROOT / "results/thesis_route_one/joint_classifier_hpo_v13"
STUDY_DIRECTORY = STUDY_ROOT / "joint/dit_classifier" / DATASET / "joint_dit_classifier"
TENSORBOARD_DIRECTORY = STUDY_DIRECTORY / "tensorboard"

In [ ]:
from IPython.display import Markdown, display


%load_ext tensorboard
%tensorboard --logdir "$TENSORBOARD_DIRECTORY" --port 6006 --bind_all
display(Markdown("Local JupyterLab/Docker: [Open TensorBoard](http://localhost:6006)."))

## 2. Run or continue

- V1 jointly trains the diffusion model and classifier; V2 is excluded and EMA remains off.
- Classifier training searches `clf_train_batch_fraction=[0.0, 0.25, 0.5]`,
  `clf_train_noisy_input_type=["noisy", "clean"]`, and
  `clf_train_class_input_type=["null_class_only", "all_classes"]`: 12 combinations.
- Fraction zero retains the full-batch forward paths. Positive fractions reserve random classifier rows
  and use the remaining rows for denoising in one shared student pass; each loss averages its own rows.
  Both row masks are disabled. Image/timestep and conditioning choices transform the classifier inputs.
  Internal features are used (`aggregate_from_noises=False`); `clf_loss_coef=1.0` is fixed.
- All trials report **ordinary raw classifier accuracy on clean images with null conditioning**. No ensemble is computed.
  V2 classifier train/test noising-cap parameters remain absent from the profile.
- No validation runs during fitting (`validation_freq=[]`); training history contains no `val_*` metrics.
  Final evaluation is retained for both Optuna objectives and uses the final-epoch raw model.
  Early stopping is disabled (`patience=0`).
  Plateau LR reduction remains disabled (`plateau_jump=False`, `reduce_lr_patience=0`),
  and no `clipnorm` is supplied. Neither objective is hidden in a weighted HPO score.
- Final denoising scores retain native default timesteps 0-999 with fixed random draws per split.
  These bounds belong to the separate denoising metric; classifier evaluation is clean.
  TMCL transfer applies the registered continual evaluation policy separately.
  `modify_first_t` is fixed at its native default `False`. Final evaluation preserves training RNG state.
- NaN/Inf losses or final objectives prune a trial. OOM is recorded as failed.
  Finite trials run the full epoch budget; ordinary accuracy-based pruning is disabled.
- Every completed trial saves weights, CSVs, history plots, TensorBoard logs and one fixed-seed
  raw image grid (`quick_scale3`: 50 steps, CFG scale 3, eta 0). Long generation trajectories
  and GIF rendering are disabled. Shortlisted checkpoints can be reported separately.

In [ ]:
study = run_hpo(
    task="joint", 
    model_name="dit_classifier", 
    dataset_name=DATASET, 
    validation_source=VALIDATION_SOURCE, 
    validation_ratio=VALIDATION_RATIO, 
    search_profile="joint_dit_classifier", 
    n_trials=TOTAL_TRIALS, 
    concurrent_trials=CONCURRENT_TRIALS,
    worker_gpu_memory_limit_mb=WORKER_GPU_MEMORY_LIMIT_MB,
    trial_budget_mode="total", 
    n_startup_trials=STARTUP_TRIALS, 
    epochs=EPOCHS, 
    seed=SEED, 
    results_path=str(STUDY_ROOT), 
    timeout=None if TIMEOUT_HOURS is None else TIMEOUT_HOURS * 3600, 
    objective_metrics=["classification_accuracy", "noise_loss"], 
    objective_directions=["maximize", "minimize"], 
    dtype_policy=DTYPE_POLICY, 
)

## 3. Pareto results

Each row is a completed trial that no other trial improves in both objectives.
Accuracy is a fraction; lower noise loss is better. There is no single automatic winner.
Inspect generated images too: noise MSE is not a perceptual-quality metric.
`summarize_hpo(study, pareto_only=False)` includes failed/pruned attempts.

In [ ]:
summarize_hpo(study)

## 4. TensorBoard

Epoch metrics and both final objectives are saved per V1 trial.
The study directory also contains `trials.csv`, `pareto_trials.csv`, SQLite state and trial configurations.
Each successful run has `model.weights.h5`, evaluation/history CSVs, history plots and one quick image grid.
Per-trial 1000-step generation modes and GIF reports are disabled.
Aborted trials retain their available logs/configuration and divergence evidence, not fabricated final artifacts.
Concurrent workers also retain a per-trial process log for training output and failures.